This is the same querychat you saw Monday. Today we'll dissect how it actually works and learn some key llm concepts.

In [ ]:
TIPS_URL = "https://raw.githubusercontent.com/mwaskom/seaborn-data/master/tips.csv"
df = pd.read_csv(TIPS_URL)

qc = QueryChat(
    df,
    "tips",
    client=ctl.ChatGithub(model="gpt-4.1-mini"),
    # client=ctl.ChatGithub(model="gpt-4.1"),
    # client=ctl.ChatAnthropic(),
)

In [ ]:
client_demo = qc.client()
client_demo.chat("What day had the highest average tip?")

# querychat: What the LLM Sees

`QueryChat(df, "tips")` gives us a good opportunity to talk about almost all main concepts we need to discuss:

- System prompt customization
- Data schema 
- Enforcing structured output
- Tool calling 
- Behavioral constraints / Guardrails

Moreover, it gives us a great example how we can prototype a LOT of stuff for weeks 3/4 outside of shiny app and parallelize work.

querychat is built on top of chatlas, so we can use its method (on data steriods!) from our apps and scripts

In [ ]:
import chatlas as ctl
import duckdb
import pandas as pd
from dotenv import load_dotenv
from querychat import QueryChat

load_dotenv()

---
## A. System Prompt (Role & Persona)

📎 [Slides: LLM Quick Start](https://dsci532-review-pages.netlify.app/slides/07-llm-dev-a.html)

In [ ]:
print(qc.system_prompt)

Let us dig in into how this prompt is structured. Let's try to make sense of it [here](https://dsci532-prompt-blocks.netlify.app/prompt-blocks-activity.html#/answers)

---
## B. Data Schema (What the LLM Knows)

📎 [Slides: LLM Quick Start](https://dsci532-review-pages.netlify.app/slides/07-llm-dev-a.html)

Inside `<database_schema>` tags, the LLM sees:
- Column names and types (`FLOAT`, `TEXT`, `INTEGER`)
- Value ranges (`total_bill`: 3.07–50.81)
- Categorical values (`day`: Sun, Sat, Thur, Fri)

**Crucially: by default, no actual data rows.** The LLM knows *what* the data looks
like but has not *seen* any of it. It can only access rows by calling tools.

---
## C. Structured Output (SQL Constraints)

📎 [Slides: Structured Output](https://dsci532-review-pages.netlify.app/slides/08b-structured.html)

Block 3 in the prompt constrains *how* the LLM writes SQL:
- Single `SELECT` only — no `DROP`, `CREATE`, `ALTER`
- No trailing semicolons
- All computed columns must be aliased

This is **structured output** — the LLM doesn't return free text,
it returns SQL that must match a specific format.

---

> **See also:**
> - [`structured/01-simple.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/01-simple.py) — structured output via Pydantic (`chat.chat_structured(..., data_model=Person)`)
> - [`structured/02-image.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/02-image.py) — structured output from images (shape + colour extraction)
> - [`structured/03-pdf.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/03-pdf.py) / [`03-pdf-tokens.ipynb`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/03-pdf-tokens.ipynb) — structured output from PDFs (Likert scale tables)

---
## D. Tool Calling

📎 [Slides: Tool Calling](https://dsci532-review-pages.netlify.app/slides/08a-tools.html)

### How tool calling does NOT work

Common misconception: the LLM calls external APIs directly.

```mermaid
sequenceDiagram
    participant U as User
    participant A as Assistant (LLM)
    participant W as openweather.org

    U->>A: {role: "user", content: "What's the weather at Fenway Park?"}
    A->>W: GET /weather/current/us/02215.json
    W-->>A: {"temp": 65, "conditions": "sunny"}
    A->>U: {role: "assistant", content: "It's 65° and sunny—a beautiful day!"}
```

The LLM **does not make HTTP requests** (* in web interfaces there is also a tool layer LLM can ask to do that). It's a text-in, text-out model
running on a remote server — it has no network access to the outside world.

### How tool calling DOES work

The **client code** (your Python script) executes the tool on behalf of the LLM:

```mermaid
sequenceDiagram
    participant W as openweather.org
    participant U as User (client code)
    participant A as Assistant (LLM)

    U->>A: {role: "user", content: "Weather at Fenway Park?", tools: [get_current_weather]}
    A->>U: CALL get_current_weather("02215")
    U->>W: GET /weather/current/us/02215.json
    W-->>U: {"temp": 65, "conditions": "sunny"}
    U->>A: RETURN {"temp": 65, "conditions": "sunny"}
    A->>U: {role: "assistant", content: "It's 65° and sunny—a beautiful day!"}
```

Key insight: We tell LLM what tools are available, LLM only **asks** for a tool call. It's the client (chatlas,
your script, shiny app code) that actually **runs** the function and sends the result back.

This is exactly what `chat.register_tool()` sets up, and what querychat
does with `querychat_query` and `querychat_update_dashboard` below.

### querychat's two tools

querychat gives the LLM **two tools**. The data flow is different for each:

#### `querychat_update_dashboard` — NO data flows to the LLM

```mermaid
sequenceDiagram
    participant S as Shiny App
    participant LLM
    participant DB as Shiny App (DuckDB)
    participant UI as Shiny UI

    S->>LLM: "Show me only dinner parties of 4+"
    LLM->>S: REQUEST update_dashboard(sql="SELECT * ... WHERE ...", title="...")
    S->>DB: Execute SQL
    DB-->>UI: Filtered rows → DataGrid
    S-->>LLM: "Dashboard updated."
    Note over LLM: LLM never sees the data!
    LLM-->>S: "Done — filtered to dinner parties of 4+."
```

#### `querychat_query` — data flows TO the LLM

```mermaid
sequenceDiagram
    participant S as Shiny App
    participant LLM
    participant DB as Shiny App (DuckDB)

    S->>LLM: "What is the average tip by day?"
    LLM->>S: REQUEST querychat_query(sql="SELECT day, AVG(tip)...")
    S->>DB: Execute SQL
    DB-->>S: [{day: Sun, avg: 3.26}, ...]
    S-->>LLM: JSON rows [{day: Sun, avg: 3.26}, ...]
    Note over LLM: LLM sees actual data!
    LLM-->>S: "The average tip on Sunday is $3.26..."
```

Let's watch both loops in action using `on_tool_request` / `on_tool_result` callbacks.

In [ ]:
client = qc.client()

step = [0]


def show_request(req):
    step[0] += 1
    print(f"\n── Step {step[0]}: LLM requests tool ─────────────────")
    print(f"   Tool:      {req.name}")
    print(f"   Arguments: {req.arguments}")


def show_result(res):
    step[0] += 1
    value_str = str(res.value)[:300]
    print(f"\n── Step {step[0]}: Tool returns result ──────────────────")
    print(f"   Sent back to LLM: {value_str}")


client.on_tool_request(show_request)
client.on_tool_result(show_result)

In [ ]:
step[0] = 0
print("── Step 0: User sends message ──────────────────────")
print('   "What is the average tip by day?"')

response = client.chat("What is the average tip by day?", echo="none")

step[0] += 1
print(f"\n── Step {step[0]}: LLM responds to user ─────────────────")
print(f"   {response}")

The query tool returned **actual data rows** to the LLM — it read them
and produced a human-readable summary. This is why `querychat_query`
is a **data exposure** risk.

Now let's see the **update** tool — same flow, but the LLM never sees data:

In [ ]:
# Fresh client with update_dashboard callback
dashboard_sql = {"query": None, "title": None}


def on_update(data):
    dashboard_sql.update(data)


client2 = qc.client(update_dashboard=on_update)
step[0] = 0
client2.on_tool_request(show_request)
client2.on_tool_result(show_result)

print("── Step 0: User sends message ──────────────────────")
print('   "Show me only dinner parties of 4 or more"')

response = client2.chat("Show me only dinner parties of 4 or more", echo="none")

step[0] += 1
print(f"\n── Step {step[0]}: LLM responds to user ─────────────────")
print(f"   {response}")

The tool result was `"Dashboard updated."` — **not data rows**.
The LLM generated SQL but never saw the filtered data.

Meanwhile, our callback captured the SQL. In Shiny, this is where
the UI would update the DataGrid:

In [ ]:
# This is what Shiny would do with the callback
print(f"Title: {dashboard_sql['title']}")
print(f"SQL:   {dashboard_sql['query']}")
print()

if dashboard_sql["query"]:
    con = duckdb.connect()
    con.register("tips", df)
    print(con.execute(dashboard_sql["query"]).fetchdf())
    con.close()

### Comparing the two tools

| | `querychat_query` | `querychat_update_dashboard` |
|---|---|---|
| **Purpose** | Answer a question | Filter the dashboard |
| **LLM generates** | SQL query | SQL query + title |
| **Tool returns** | JSON data rows | `"Dashboard updated."` |
| **LLM sees data?** | **Yes** | **No** |
| **Privacy safe?** | No | Yes |

This is why `tools="update"` is the **privacy mode** — the LLM can write SQL
to filter your dashboard, but never sees a single data value.

---

> **See also:**
> - [`tools/00-capital-finder.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/00-capital-finder.py) — basic tool calling
>   (system prompt controls *whether* the LLM calls a tool)
> - [`tools/chatlas-weather.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/chatlas-weather.py) — multi-tool chain (geocoding → weather API)
> - [`tools/app-weather-core.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/app-weather-core.py) / [`app-weather-express.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/app-weather-express.py) — tool calling inside a Shiny app

---
## E. Behavioral Constraints & Customization

📎 [Slides: LLM Quick Start](https://dsci532-review-pages.netlify.app/slides/07-llm-dev-a.html)

Blocks 6–7 in the system prompt constrain **how** the LLM behaves:
- **Suggestions** (Block 6): rules for generating clickable prompt buttons
- **Guidelines** (Block 7): "never use prior knowledge", "be concise", "use Markdown tables"

You can add your own constraints via three customization levers:

| Lever | Where it goes | Purpose |
|-------|--------------|--------|
| `greeting` | Chat UI (first message) | Welcome text the user sees |
| `data_description` | System prompt | Context about the dataset |
| `extra_instructions` | System prompt | Formatting rules, behavior |

### Greeting

By default, querychat asks the LLM to generate a greeting (costs 1 API call).
You can provide a static string instead — free, fast, deterministic.

In [ ]:
# Default: LLM generates the greeting (costs 1 API request)
print("── LLM-generated greeting ──────────────────────────")
print(qc.generate_greeting())

In [ ]:
# Static greeting: free, instant, deterministic
qc_with_greeting = QueryChat(
    df,
    "tips",
    client=ctl.ChatGithub(model="gpt-4.1-mini"),
    greeting="Welcome! Ask me about restaurant tipping — by day, party size, or smoker status.",
)

print("── Static greeting (no LLM call) ───────────────────")
print(qc_with_greeting.greeting)

### data_description & extra_instructions

These go into the **system prompt** — they change how the LLM
reasons about data and formats its answers.

In [ ]:
qc_custom = QueryChat(
    df,
    "tips",
    client=ctl.ChatGithub(model="gpt-4.1-mini"),
    data_description=(
        "Restaurant tipping data collected by a waiter over several months. "
        "Includes meal cost, tip amount, party size, and demographics."
    ),
    extra_instructions=(
        "Always show tip as a percentage of total_bill. "
        "Format dollar amounts with $ and two decimals. "
        "Keep responses to 2-3 sentences."
    ),
)

# See what changed in the prompt
prompt = qc_custom.system_prompt
for tag in ["data_description", "Additional Instructions"]:
    idx = prompt.find(tag)
    if idx >= 0:
        print(prompt[max(0, idx - 20):idx + 200])
        print("...\n")

In [ ]:
client_custom = qc_custom.client()
response = client_custom.chat("Compare smokers vs non-smokers", echo="none")
print(response)

---
## Summary

```mermaid
graph LR
    QC["QueryChat(df, 'tips')"] --> A["A. Role preamble"]
    QC --> B["B. Data schema"]
    QC --> C["C. SQL constraints"]
    QC --> D["D. Tool definitions"]
    QC --> E["E. Behavioral rules"]
    QC --> G["greeting"]

    A --> SP[System Prompt]
    B --> SP
    C --> SP
    D --> SP
    E --> SP
    G --> UI[Chat UI]

    style SP fill:#2d6a4f,color:#fff
    style UI fill:#1d3557,color:#fff
```

| Method | What you get |
|--------|-------------|
| `qc.system_prompt` | Full prompt the LLM sees (A–E) |
| `qc.generate_greeting()` | LLM-generated welcome (costs 1 call) |
| `QueryChat(..., greeting="...")` | Static greeting (free, instant) |
| `QueryChat(..., data_description="...")` | Context about dataset → system prompt |
| `QueryChat(..., extra_instructions="...")` | Formatting/behavior rules → system prompt |
| `qc.client()` | Chat client with all tools |
| `qc.client(tools="update")` | Privacy mode — no data queries |
| `client.on_tool_request(fn)` | Intercept each tool call |
| `client.on_tool_result(fn)` | Intercept each tool result |

### Demos (there are more in code folder)!

| File | Concept |
|------|--------|
| [`tools/00-capital-finder.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/00-capital-finder.py) | Tool calling basics, system prompt steers tool use |
| [`structured/01-simple.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/01-simple.py) | Structured output (Pydantic) from text |
| [`structured/02-image.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/02-image.py) | Structured output from images |
| [`structured/03-pdf.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/03-pdf.py) / [`03-pdf-tokens.ipynb`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/structured/03-pdf-tokens.ipynb) | Structured output from PDFs, token costs |
| [`tools/chatlas-weather.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/chatlas-weather.py) | Multi-tool chain (geocoding → weather) |
| [`tools/app-weather-core.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/app-weather-core.py) / [`app-weather-express.py`](https://github.com/UBC-MDS/DSCI_532_vis-2_book/blob/main/code/lecture06/tools/app-weather-express.py) | Tool calling inside a Shiny app |

### Quick example

In [ ]:
client = qc.client()
client.chat("What is the average tip by day?")

---
## Other Topics

querychat covers **tool calling** and **structured output**, but production LLM apps
often need more. Three topics we'll explore separately:

### MCP (Model Context Protocol)
A standard protocol for connecting LLMs to **external tools and data sources** —
databases, APIs, file systems — without writing custom integration code.
Instead of registering one tool at a time (like `chat.register_tool()`),
MCP lets you plug in a whole **server** of tools that the LLM can discover
and call.

📎 [Slides: MCP](https://dsci532-review-pages.netlify.app/slides/08d-mcp.html)

### RAG (Retrieval-Augmented Generation)
Instead of stuffing all context into the system prompt (which hits the
context window limit), RAG **retrieves** only the relevant chunks at query
time. Think of it as a smarter version of `data_description` — but for
thousands of documents.

📎 [Slides: RAG](https://dsci532-review-pages.netlify.app/slides/08c-rag.html)

### Evals (Evaluation)
How do you know your LLM app actually works? Evals are **automated tests
for LLM outputs** — measuring accuracy, safety, and consistency across
many inputs. Essential before deploying anything built with chatlas or
querychat.

📎 [Slides: Evals & Inspect](https://dsci532-review-pages.netlify.app/slides/08e-evals_inspect.html)